In [1]:
import jsonlines
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.linear_model import LogisticRegression
import numpy as np
import nltk
from nltk.tag import pos_tag
from nltk.tokenize import word_tokenize
import regex as re

In [2]:
data_list = []

In [3]:
with jsonlines.open("data/train.jsonl", mode='r') as reader:
    for obj in reader:
        obj.pop('model')
        obj.pop('genre')
        obj.pop('id')
        data_list.append(obj)

In [4]:
data_list[:5]

[{'text': 'Duke Ellington, a titan of jazz, revolutionized the genre through his innovative compositions, showcasing a remarkable ability to integrate voice and instrumental music. Among the notable figures who contributed to this artistic symphony was Ivie Anderson, whose scat singing mirrored the improvisational prowess of musicians like Nanton. In "Ring Dem Bells," Ellington ingeniously interweaves scat singing as a dialogue with saxophones, creating a dynamic call and response that underscores his vision of music as a fluid conversation between voices and instruments.\n\nEllington consistently emphasized the voice as an instrument of equal importance to traditional brass and woodwinds, orchestrating his compositions with a keen ear for vocal qualities. This approach is vividly demonstrated in "Mood Indigo," where his orchestration skills shine through non-traditional chord arrangements, transforming the piece into an auditory tapestry of mood and color. It is not merely the notes t

In [5]:
len(data_list)

23707

In [6]:
df = pd.DataFrame(data_list)

In [7]:
print(df)

                                                    text  label
0      Duke Ellington, a titan of jazz, revolutionize...      1
1      I reflected on the shifting dynamics of media ...      1
2      In F. Scott Fitzgerald's "The Great Gatsby," t...      1
3      I still chuckle when I think about that time I...      1
4      Yoga, originating in ancient India, encompasse...      1
...                                                  ...    ...
23702  Antioch stands as a pivotal point in the narra...      1
23703  Margaret Cavendish’s The Blazing World stands ...      1
23704  In August of 1793, in a last-ditch effort to w...      0
23705  In interacting with Luciente and others, Conni...      0
23706  Meursault and Clamence, the protagonists from ...      1

[23707 rows x 2 columns]


In [8]:
print(df.text[0])

Duke Ellington, a titan of jazz, revolutionized the genre through his innovative compositions, showcasing a remarkable ability to integrate voice and instrumental music. Among the notable figures who contributed to this artistic symphony was Ivie Anderson, whose scat singing mirrored the improvisational prowess of musicians like Nanton. In "Ring Dem Bells," Ellington ingeniously interweaves scat singing as a dialogue with saxophones, creating a dynamic call and response that underscores his vision of music as a fluid conversation between voices and instruments.

Ellington consistently emphasized the voice as an instrument of equal importance to traditional brass and woodwinds, orchestrating his compositions with a keen ear for vocal qualities. This approach is vividly demonstrated in "Mood Indigo," where his orchestration skills shine through non-traditional chord arrangements, transforming the piece into an auditory tapestry of mood and color. It is not merely the notes that define El

In [9]:
tokens_all = [word_tokenize(sentence) for sentence in df.text]

In [ ]:
# Removing all tokens consisting only of non-alphanumeric characters, that aren't inbetween words e.g. "battle-field",
# so any tokens such as: "''", "--", ".", "," etc. are removed, but "battle-field" is not removed. 
for sentence in tokens_all:
    while True:
        for token in sentence:
            if re.search(r"\B[^a-zA-Z0-9]+\B", token) != None:
                sentence.remove(token) # For each token in a sentence, if the token consists only of 1 or more non-alphanumeric characters, remove it from the sentence.

        outs = [] # Creates an empty list. The list is reset to empty every iteration of the while-loop. 
        for token in sentence:
            if re.search(r"\B[^a-zA-Z0-9]+\B", token) != None:
                outs.append(token) # This for-loop makes an additional check for any non-alphanumeric tokens, and appends then to the empty list 'outs'.

        if outs: # This checks whether the list 'outs' is empty or not. If it's not empty, then 'continue' is activated and the while-loop continues to iterate,
            continue # if the list is empty, then 'break' is activated, and the while-loop terminates, and the outer-most for-loop continues to the next 'sentence' in 'tokens_all'.
        else:
            break

In [121]:
print(tokens_all[15000])

['She', 'was', 'small', 'almost', 'childish', 'in', 'stature', 'with', 'a', 'thin', 'erect', 'little', 'figure', 'and', 'a', 'pale', 'oval', 'face', 'framed', 'in', 'short', 'curly', 'hair', 'and', 'at', 'first', 'sight', 'people', 'always', 'called', 'her', 'plain', 'an', 'insignificant', 'puny', 'little', 'thing', 'that', 'was', 'what', 'they', 'said', 'until', 'they', 'saw', 'her', 'eyes', 'and', 'they', 'were', 'the', 'most', 'wonderful', 'and', 'spirituelle', 'eyes', 'in', 'the', 'world', 'And', 'after', 'that', 'they', 'were', 'not', 'so', 'sure', 'of', 'the', 'plainness', 'For', 'comparisons', 'are', 'odious', 'and', 'there', 'is', 'no', 'hard', 'and', 'fast', 'rule', 'with', 'respect', 'to', 'feminine', 'beauty', 'at', 'least', 'tastes', 'differ', 'and', 'here', 'and', 'there', 'a', 'Philistine', 'might', 'be', 'found', 'who', 'would', 'be', 'ready', 'to', 'swear', 'that', 'dark', 'spirituelle', 'eyes', 'brimful', 'of', 'intelligence', 'and', 'animation', 'with', 'a', 'mirthful

In [122]:
words = [[word.lower() for word in sentence] for sentence in tokens_all]

In [127]:
len(words)

23707

In [132]:
print(words[-1])

['meursault', 'and', 'clamence', 'the', 'protagonists', 'from', 'albert', 'camus', "'s", 'the', 'stranger', 'and', 'patrick', 'kertes', "'s", 'the', 'tenant', 'respectively', 'embody', 'starkly', 'differing', 'attitudes', 'towards', 'isolation', 'and', 'relationships', 'meursault', 'the', 'unassuming', 'and', 'detached', 'protagonist', 'of', 'the', 'stranger', 'is', 'largely', 'unaware', 'of', 'his', 'isolation', 'from', 'society', 'this', 'is', 'evident', 'when', 'he', 'states', 'the', 'sun', 'was', 'the', 'same', 'as', 'is', 'the', 'air', 'and', 'the', 'light', 'and', 'it', 'made', 'me', 'so', 'happy', 'as', 'for', 'the', 'other', 'things', 'i', 'was', 'careful', 'to', 'avoid', 'them', 'camus', '1', 'meursault', "'s", 'indifference', 'stems', 'from', 'a', 'profound', 'existential', 'isolation', 'which', 'continues', 'throughout', 'his', 'life', 'he', 'lives', 'in', 'a', 'state', 'of', 'absurdity', 'disassociated', 'from', 'societal', 'norms', 'and', 'expectations', 'displaying', 'a',

In [124]:
words_list = []

In [125]:
for sentence in words:
    for word in sentence:
        words_list.append(word)

In [126]:
len(words_list)

14949644

In [131]:
print(words_list[14949644-1])

condition


In [133]:
vocab = sorted(set(words_list))

In [134]:
len(vocab)

122750

In [150]:
print(vocab[5000:5020])

['affrighted', 'affrightedly', 'affrighting', 'affront', 'affronted', 'affronting', 'affronts', 'afghan', 'afghan-american', 'afghan-led', 'afghani', 'afghanistan', 'afghans', 'afi', 'afia', 'aficionado', 'aficionados', 'afield', 'afire', 'afixture']


In [20]:
tags = pos_tag(tokens)

In [21]:
print(tags)

[('Duke', 'NNP'), ('Ellington', 'NNP'), ('a', 'DT'), ('titan', 'NN'), ('of', 'IN'), ('jazz', 'NN'), ('revolutionized', 'VBN'), ('the', 'DT'), ('genre', 'NN'), ('through', 'IN'), ('his', 'PRP$'), ('innovative', 'JJ'), ('compositions', 'NNS'), ('showcasing', 'VBG'), ('a', 'DT'), ('remarkable', 'JJ'), ('ability', 'NN'), ('to', 'TO'), ('integrate', 'VB'), ('voice', 'NN'), ('and', 'CC'), ('instrumental', 'JJ'), ('music', 'NN'), ('Among', 'IN'), ('the', 'DT'), ('notable', 'JJ'), ('figures', 'NNS'), ('who', 'WP'), ('contributed', 'VBD'), ('to', 'TO'), ('this', 'DT'), ('artistic', 'JJ'), ('symphony', 'NN'), ('was', 'VBD'), ('Ivie', 'NNP'), ('Anderson', 'NNP'), ('whose', 'WP$'), ('scat', 'NN'), ('singing', 'NN'), ('mirrored', 'VBD'), ('the', 'DT'), ('improvisational', 'JJ'), ('prowess', 'NN'), ('of', 'IN'), ('musicians', 'NNS'), ('like', 'IN'), ('Nanton', 'NNP'), ('In', 'IN'), ('Ring', 'NNP'), ('Dem', 'NNP'), ('Bells', 'NNP'), ('Ellington', 'NNP'), ('ingeniously', 'RB'), ('interweaves', 'VBZ'),

In [23]:
type(tags[0])

tuple

In [14]:
tfidf = TfidfVectorizer(sublinear_tf=True, min_df=5, norm='l2', encoding="utf-8", ngram_range=(1, 2), stop_words='english')

features = tfidf.fit_transform(df.text).toarray()
labels = df.label

In [ ]:
features.shape # Now each of the 23707 sequences of text is represented by 195904 features, 
               # representing the tf-idf score for different unigrams and bigrams.

(23707, 195904)

In [28]:
model = LogisticRegression(random_state=42)

In [29]:
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=42)

In [31]:
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [34]:
print(metrics.classification_report(y_test, y_pred, labels=df['label']))

              precision    recall  f1-score   support

           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           0       0.97      0.96      0.97      1856
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97      0.98      0.98      2886
           1       0.97    